In [1]:
# ============================================================================
# scVI LATENT EMBEDDING EXTRACTION  (Python — Notebook 2 / 2)
#
# Input : filtered count CSVs produced by scvi-gene-export.ipynb  (R notebook)
#         → dataset/ST/scVI_counts/{sample}.csv
# Output: per-spot .pt files (50-d float32 tensor, one file per QC-passed patch)
#         → dataset/Gene Embedding Extraction/scvi_latent_pt_embeddings/{sample}/
#
# Install once if running locally:
#   pip install scvi-tools scanpy anndata tqdm
# ============================================================================
import os, torch
import numpy as np
import pandas as pd
import anndata as ad
import scvi
import scanpy as sc
from pathlib import Path
from tqdm import tqdm

print(f'scvi-tools : {scvi.__version__}')
print(f'torch      : {torch.__version__}')
print(f'GPU        : {torch.cuda.is_available()}')

scvi-tools : 1.4.3
torch      : 2.3.1
GPU        : True


In [2]:
# ============================================================================
# CONFIG — paths auto-switch between local and Kaggle
#
# KAGGLE NOTE: The filtered count CSVs (from the R notebook) must be available
# at COUNTS_DIR. On Kaggle, upload them as a separate dataset first, or run
# the R notebook in the same session and adjust COUNTS_DIR to /kaggle/working/scVI_counts.
# ============================================================================
IS_KAGGLE = os.path.exists('/kaggle/working')

if IS_KAGGLE:
    _KG        = '/kaggle/input/datasets/wanianaeem/zenodo-pt-and-hm-dataset'
    # If you uploaded the filtered CSVs as a separate dataset, change COUNTS_DIR:
    COUNTS_DIR = '/kaggle/working/scVI_counts'   # <- adjust if uploaded as dataset
    COORDS_CSV = f'{_KG}/spot_spatial_coordinates.csv'
    PNG_ROOT   = f'{_KG}/.png patches/.png patches'
    OUTPUT_DIR = '/kaggle/working/scvi_latent_pt_embeddings'
else:
    # Notebook lives at ST_Project/03_Embedding_Extraction/Gene/
    # Two .parent calls → ST_Project/ root
    _ROOT      = Path.cwd().parent.parent
    COUNTS_DIR = str(_ROOT / 'dataset/ST/scVI_counts')
    COORDS_CSV = str(_ROOT / 'Outputs/Patient-Sample-Information/spot_spatial_coordinates.csv')
    PNG_ROOT   = str(_ROOT / 'dataset/.png patches/.png patches')
    OUTPUT_DIR = str(_ROOT / 'dataset/Gene Embedding Extraction/scvi_latent_pt_embeddings')

SAMPLES = [
    'IU_PDA_HM11', 'IU_PDA_HM13',
    'IU_PDA_T1',   'IU_PDA_T3',
    'IU_PDA_T4',   'IU_PDA_T11',
]

# ---- scVI hyperparameters ----
N_LATENT   = 50    # latent dim — must match gene branch input dim in Phase A
N_HIDDEN   = 128
N_LAYERS   = 2
N_EPOCHS   = 400
BATCH_SIZE = 256
SEED       = 42

print(f'Running : {"Kaggle" if IS_KAGGLE else "local"}')
print(f'Counts  : {COUNTS_DIR}')
print(f'Output  : {OUTPUT_DIR}')

Running : local
Counts  : c:\Users\datai\Downloads\ST_Project\dataset\ST\scVI_counts
Output  : c:\Users\datai\Downloads\ST_Project\dataset\Gene Embedding Extraction\scvi_latent_pt_embeddings


In [3]:
# ============================================================================
# CELL 3 — Load QC-filtered count CSVs → combined AnnData
#
# CSV layout (from R write.csv): genes as rows, barcodes as columns.
# AnnData convention is the transpose: obs = spots, var = genes.
# ============================================================================
adata_list = []

for sample in SAMPLES:
    csv_path = os.path.join(COUNTS_DIR, f'{sample}.csv')
    if not os.path.exists(csv_path):
        print(f'  MISSING: {csv_path}')
        continue

    df = pd.read_csv(csv_path, index_col=0)   # genes × barcodes
    print(f'{sample}: {df.shape[0]} genes × {df.shape[1]} barcodes (QC-filtered)')

    adata = ad.AnnData(X=df.T.values.astype(np.float32))
    adata.obs_names = df.columns.tolist()      # original barcodes (PDACH_XX_...)
    adata.var_names = df.index.tolist()        # gene names
    adata.obs['sample'] = sample               # batch label for scVI
    adata_list.append(adata)

# IMPORTANT: do NOT use keys= here — it would mangle obs_names and break
# the barcode lookup in Cell 6.  Each sample already has a unique chip
# prefix (PDACH_10_, PDACH_12_, …) so barcodes are globally unique.
adata_all = ad.concat(adata_list, join='inner')
adata_all.obs_names_make_unique()   # safety; typically no-op here

# Stash raw integer counts before any normalisation — scVI needs them
adata_all.layers['counts'] = adata_all.X.copy()

print(f'\nCombined: {adata_all.n_obs} spots × {adata_all.n_vars} genes')
print(adata_all.obs['sample'].value_counts().to_string())

IU_PDA_HM11: 17893 genes × 3894 barcodes (QC-filtered)
IU_PDA_HM13: 17893 genes × 1387 barcodes (QC-filtered)
IU_PDA_T1: 17893 genes × 3073 barcodes (QC-filtered)
IU_PDA_T3: 17893 genes × 4241 barcodes (QC-filtered)
IU_PDA_T4: 17893 genes × 3587 barcodes (QC-filtered)
IU_PDA_T11: 17893 genes × 2677 barcodes (QC-filtered)

Combined: 18859 spots × 17893 genes
sample
IU_PDA_T3      4241
IU_PDA_HM11    3894
IU_PDA_T4      3587
IU_PDA_T1      3073
IU_PDA_T11     2677
IU_PDA_HM13    1387


In [4]:
# ============================================================================
# CELL 4 — Select highly variable genes (HVG)
#
# Normalise only to select HVGs; restore raw counts afterwards so scVI
# receives the integer values it expects.
# ============================================================================
sc.pp.normalize_total(adata_all, target_sum=1e4)
sc.pp.log1p(adata_all)

sc.pp.highly_variable_genes(
    adata_all,
    n_top_genes=3000,
    subset=False,
    flavor='seurat_v3',
    batch_key='sample',
    layer='counts',       # seurat_v3 HVG selection uses raw counts
)

print(f'HVGs selected: {adata_all.var["highly_variable"].sum()}')

# Restore raw counts as the main matrix
adata_all.X = adata_all.layers['counts'].copy()

HVGs selected: 3000


In [5]:
# ============================================================================
# CELL 5 — Train scVI
# ============================================================================
scvi.settings.seed = SEED

adata_hvg = adata_all[:, adata_all.var['highly_variable']].copy()
print(f'Training on {adata_hvg.n_obs} spots × {adata_hvg.n_vars} HVGs')

scvi.model.SCVI.setup_anndata(
    adata_hvg,
    layer='counts',
    batch_key='sample',    # correct for cross-sample batch effects
)

model = scvi.model.SCVI(
    adata_hvg,
    n_latent=N_LATENT,
    n_hidden=N_HIDDEN,
    n_layers=N_LAYERS,
)

model.train(
    max_epochs=N_EPOCHS,
    batch_size=BATCH_SIZE,
    early_stopping=True,
    early_stopping_patience=20,
    plan_kwargs={'lr': 1e-3},
)

print('Training complete.')
print(f'  Final ELBO: {model.history["elbo_train"].iloc[-1]:.2f}')

Seed set to 42


Training on 18859 spots × 3000 HVGs


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\datai\anaconda3\envs\tcga\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
c:\Users\datai\anaconda3\envs\tcga\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:434: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Training:   0%|          | 0/400 [00:00<?, ?it/s]

Monitored metric elbo_validation did not improve in the last 20 records. Best score: 1051.537. Signaling Trainer to stop.
Training complete.


TypeError: unsupported format string passed to Series.__format__

In [ ]:
print(f'  Final ELBO: {model.history["elbo_train"].iloc[-1].item():.2f}')

  Final ELBO: 1051.95


In [7]:
# ============================================================================
# CELL 6 — Extract 50-d latent embeddings; build barcode → vector lookup
# ============================================================================
latent = model.get_latent_representation(adata_hvg)   # shape [N_spots, 50]
print(f'Latent embedding shape: {latent.shape}')

# obs_names are the original barcodes (e.g. PDACH_10_AAACAAGTATCTCCCA-1)
# Verify no duplicates crept in from obs_names_make_unique()
n_unique = len(set(adata_hvg.obs_names))
assert n_unique == adata_hvg.n_obs, \
    f'Duplicate barcodes after concat ({adata_hvg.n_obs - n_unique} collisions). '\
    'Check chip prefixes across samples — they must differ.'

barcode_to_latent = {
    bc: latent[i]
    for i, bc in enumerate(adata_hvg.obs_names)
}
print(f'Lookup built: {len(barcode_to_latent)} barcodes')

Latent embedding shape: (18859, 50)
Lookup built: 18859 barcodes


In [8]:
# ============================================================================
# CELL 7 — Build (sample_name, row, col) → barcode lookup
# ============================================================================
coords = pd.read_csv(COORDS_CSV)
# rc_key = "IU_PDA_HM11_50_102"  (image + row + col)
coords['rc_key'] = (
    coords['image'].astype(str) + '_' +
    coords['row'].astype(str)   + '_' +
    coords['col'].astype(str)
)
rc_to_barcode = dict(zip(coords['rc_key'], coords['spot_barcode']))
print(f'Coordinates loaded: {len(rc_to_barcode)} entries')

for s in SAMPLES:
    n = (coords['image'] == s).sum()
    print(f'  {s}: {n} spots in coordinates CSV')

Coordinates loaded: 91496 entries
  IU_PDA_HM11: 3931 spots in coordinates CSV
  IU_PDA_HM13: 2182 spots in coordinates CSV
  IU_PDA_T1: 3530 spots in coordinates CSV
  IU_PDA_T3: 4354 spots in coordinates CSV
  IU_PDA_T4: 3621 spots in coordinates CSV
  IU_PDA_T11: 2777 spots in coordinates CSV


In [9]:
# ============================================================================
# CELL 8 — Save per-patch .pt files
#
# Iterate over PNG patches (same filenames used by the vision extractor).
# For each patch:
#   1. Parse (row, col) from filename stem.
#   2. Look up barcode via coordinates CSV.
#   3. Look up latent vector via barcode (only QC-passed spots have one).
#   4. Save tensor as {stem}.pt in the output directory.
#
# Old .pt files are deleted first to ensure no stale unfiltered embeddings.
# ============================================================================
import shutil

def parse_row_col(stem: str):
    """'IU_PDA_HM11_patch-000001_50_102' → (50, 102)"""
    parts = stem.split('_')
    return int(parts[-2]), int(parts[-1])


total_saved = total_qc_skip = total_coord_miss = 0

for sample in SAMPLES:
    png_dir = os.path.join(PNG_ROOT, sample)
    out_dir = os.path.join(OUTPUT_DIR, sample)

    if not os.path.isdir(png_dir):
        print(f'  {sample}: PNG directory not found — {png_dir}')
        continue

    # Wipe old .pt files for this sample so no unfiltered embeddings remain
    if os.path.isdir(out_dir):
        shutil.rmtree(out_dir)
    os.makedirs(out_dir, exist_ok=True)

    patches = sorted(Path(png_dir).glob('*.png'))
    print(f'\n{sample}: {len(patches)} PNG patches')

    saved = qc_skip = coord_miss = 0

    for png_path in tqdm(patches, desc=sample, leave=False):
        stem = png_path.stem
        try:
            row, col = parse_row_col(stem)
        except (IndexError, ValueError):
            coord_miss += 1
            continue

        barcode = rc_to_barcode.get(f'{sample}_{row}_{col}')
        if barcode is None:
            coord_miss += 1
            continue

        vec = barcode_to_latent.get(barcode)
        if vec is None:
            qc_skip += 1    # spot did not pass QC — expected, not an error
            continue

        torch.save(
            torch.from_numpy(vec.astype(np.float32)),
            os.path.join(out_dir, f'{stem}.pt'),
        )
        saved += 1

    print(f'  saved={saved}   qc_filtered={qc_skip}   coord_missing={coord_miss}')
    total_saved     += saved
    total_qc_skip   += qc_skip
    total_coord_miss += coord_miss

print(f'\n{"="*60}')
print(f'TOTAL SAVED        : {total_saved}')
print(f'TOTAL QC-FILTERED  : {total_qc_skip}  (correct — low-quality spots excluded)')
print(f'TOTAL COORD MISSING: {total_coord_miss}')
print(f'Output             : {OUTPUT_DIR}')


IU_PDA_HM11: 3931 PNG patches


  saved=3894   qc_filtered=37   coord_missing=0

IU_PDA_HM13: 2182 PNG patches


  saved=1387   qc_filtered=795   coord_missing=0

IU_PDA_T1: 3530 PNG patches


  saved=3073   qc_filtered=457   coord_missing=0

IU_PDA_T3: 4354 PNG patches


  saved=4241   qc_filtered=113   coord_missing=0

IU_PDA_T4: 3621 PNG patches


  saved=3587   qc_filtered=34   coord_missing=0

IU_PDA_T11: 2777 PNG patches


  saved=2677   qc_filtered=100   coord_missing=0

TOTAL SAVED        : 18859
TOTAL QC-FILTERED  : 1536  (correct — low-quality spots excluded)
TOTAL COORD MISSING: 0
Output             : c:\Users\datai\Downloads\ST_Project\dataset\Gene Embedding Extraction\scvi_latent_pt_embeddings


In [10]:
# ============================================================================
# CELL 9 — Verify .pt counts match QC-filtered spot totals
# ============================================================================
EXPECTED = {
    'IU_PDA_HM11': 3894,
    'IU_PDA_HM13': 1387,
    'IU_PDA_T1'  : 3073,
    'IU_PDA_T3'  : 4241,
    'IU_PDA_T4'  : 3587,
    'IU_PDA_T11' : 2677,
}

print(f'{"Sample":<20} {"Expected":<12} {"Actual":<12} Status')
print('-' * 55)
all_ok = True
for s in SAMPLES:
    d      = os.path.join(OUTPUT_DIR, s)
    actual = len(list(Path(d).glob('*.pt'))) if os.path.isdir(d) else 0
    exp    = EXPECTED[s]
    ok     = actual == exp
    status = 'OK' if ok else f'MISMATCH (diff={actual - exp:+d})'
    if not ok:
        all_ok = False
    print(f'{s:<20} {exp:<12} {actual:<12} {status}')

print()
print('All counts correct — ready for Phase A training.' if all_ok
      else 'Mismatch found — check QC metrics CSVs and re-run Cell 8.')

# Save the trained scVI model for reproducibility
model_dir = os.path.join(OUTPUT_DIR, '_scvi_model')
model.save(model_dir, overwrite=True)
print(f'\nscVI model saved to: {model_dir}')

Sample               Expected     Actual       Status
-------------------------------------------------------
IU_PDA_HM11          3894         3894         OK
IU_PDA_HM13          1387         1387         OK
IU_PDA_T1            3073         3073         OK
IU_PDA_T3            4241         4241         OK
IU_PDA_T4            3587         3587         OK
IU_PDA_T11           2677         2677         OK

All counts correct — ready for Phase A training.

scVI model saved to: c:\Users\datai\Downloads\ST_Project\dataset\Gene Embedding Extraction\scvi_latent_pt_embeddings\_scvi_model
